# Tách keyframe + Embedding PE-Core (Google Colab)

Notebook độc lập, không phụ thuộc code local. Chạy 4 cell theo thứ tự.

**Input:**
- Video: upload lên `Google Drive/aicv5_batch2/videos/`
- Weight PE-Core: để trống thì Colab tự tải từ HuggingFace Hub (~5GB, lần đầu mất vài phút) và lưu cache vào Drive để lần sau không phải tải lại

**Output** (lưu vào Drive để không mất khi Colab ngắt kết nối):
- `Google Drive/aicv5_batch2/keyframes/*.jpg`
- `Google Drive/aicv5_batch2/embeddings_pecore.pkl` — `{ids: [...], pe_core: [...]}`, ids dạng `video_id:frame_idx`

**Có thể ngắt và chạy lại bất cứ lúc nào** — cả 2 cell tách keyframe và embedding đều tự resume (bỏ qua phần đã xong), không phải chạy lại từ đầu.

In [ ]:
#@title 1. Cài đặt thư viện (~1 phút)
!pip install -q open_clip_torch opencv-python-headless loguru

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO GPU -- vao Runtime > Change runtime type > GPU'}")

In [ ]:
#@title 2. Mount Drive & thiết lập đường dẫn
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/aicv5_batch2')  #@param {type:"string"}

VIDEO_DIR = DRIVE_ROOT / 'videos'
KEYFRAME_DIR = DRIVE_ROOT / 'keyframes'
EMBEDDINGS_PKL = DRIVE_ROOT / 'embeddings_pecore.pkl'
HF_CACHE_DIR = DRIVE_ROOT / 'hf_cache'  # PE-Core weights cached here across sessions

for d in (VIDEO_DIR, KEYFRAME_DIR, HF_CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')

video_files = sorted([p for p in VIDEO_DIR.glob('*') if p.suffix.lower() in ('.mp4', '.mov', '.mkv', '.avi')])
print(f"VIDEO_DIR: {VIDEO_DIR}")
print(f"Tim thay {len(video_files)} video.")
if not video_files:
    print(f"\n>>> Chua co video nao. Upload video vao thu muc tren Drive:\n    {VIDEO_DIR}\n    roi chay lai cell nay.")

In [ ]:
#@title 3. Tách keyframe (feature-diff, CLIP ViT-L-14/dfn2b)
#@markdown Thuật toán giống hệt pipeline local: ngưỡng 0.6, khoảng cách tối thiểu 5 frame, lấy mẫu mỗi 5 frame gốc.
#@markdown Tự động bỏ qua video đã có keyframe rồi (resume an toàn nếu bị ngắt giữa chừng).

import cv2, time, re
import numpy as np
from PIL import Image
import open_clip

THRESHOLD = 0.6
MIN_DELTA = 5
SAMPLE_RATE = 5
BATCH_SIZE = 128

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dang tai model CLIP ViT-L-14 (dfn2b) len {device}...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained='dfn2b', device=device)
if device == 'cuda':
    clip_model = clip_model.half()
clip_model.eval()
print("CLIP san sang.")

def already_done(video_stem: str) -> bool:
    return any(KEYFRAME_DIR.glob(f"{video_stem}_frame_*.jpg"))

@torch.inference_mode()
def encode_frames(frames_bgr):
    tensors = []
    for frame in frames_bgr:
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        tensors.append(clip_preprocess(Image.fromarray(rgb)))
    batch = torch.stack(tensors).to(device)
    if device == 'cuda':
        batch = batch.half()
    feats = clip_model.encode_image(batch)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.float().cpu().numpy()

def extract_keyframes(video_path: Path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Khong mo duoc video: {video_path}")
    video_name = video_path.stem
    frame_idx = 0
    prev_feat = None
    last_kf_idx = -MIN_DELTA
    n_saved = 0
    batch_frames, batch_indices = [], []

    def flush_batch():
        nonlocal prev_feat, last_kf_idx, n_saved
        if not batch_frames:
            return
        feats = encode_frames(batch_frames)
        for i, idx in enumerate(batch_indices):
            f = feats[i]
            if prev_feat is None:
                keep = True
            else:
                diff = np.linalg.norm(f - prev_feat) / max(np.linalg.norm(prev_feat), 1e-8)
                keep = diff > THRESHOLD and (idx - last_kf_idx) >= MIN_DELTA
            if keep:
                out_path = KEYFRAME_DIR / f"{video_name}_frame_{idx:06d}.jpg"
                cv2.imwrite(str(out_path), batch_frames[i])
                prev_feat = f
                last_kf_idx = idx
                n_saved += 1
        batch_frames.clear()
        batch_indices.clear()

    t0 = time.time()
    total_frames = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % SAMPLE_RATE == 0:
            batch_frames.append(frame)
            batch_indices.append(frame_idx)
            if len(batch_frames) >= BATCH_SIZE:
                flush_batch()
        frame_idx += 1
        total_frames += 1
    flush_batch()
    cap.release()
    elapsed = time.time() - t0
    fps = total_frames / elapsed if elapsed > 0 else 0
    print(f"  {video_name}: {n_saved} keyframes tu {total_frames} frames trong {elapsed:.1f}s ({fps:.0f} fps)")
    return n_saved

todo = [v for v in video_files if not already_done(v.stem)]
print(f"Da xong: {len(video_files) - len(todo)}/{len(video_files)}. Con lai: {len(todo)}\n")

for i, vp in enumerate(todo):
    print(f"[{i+1}/{len(todo)}] {vp.name}")
    try:
        extract_keyframes(vp)
    except Exception as e:
        print(f"  LOI (bo qua, se thu lai neu chay lai cell nay): {e}")

print("\n=== XONG tach keyframe ===")

In [ ]:
#@title 4. Embedding PE-Core cho các keyframe
#@markdown Tải model `hf-hub:timm/PE-Core-bigG-14-448` (1280-dim, khớp với cấu hình local).
#@markdown Nếu đã có cache trong `hf_cache/` trên Drive từ lần chạy trước thì load nhanh, không cần tải lại.

import pickle, re, time
from collections import defaultdict

print("Dang tai PE-Core (hf-hub:timm/PE-Core-bigG-14-448)...")
pecore_model, _, pecore_preprocess = open_clip.create_model_and_transforms(
    'hf-hub:timm/PE-Core-bigG-14-448', device=device
)
if device == 'cuda':
    pecore_model = pecore_model.half()
pecore_model.eval()
print("PE-Core san sang.")

# Load existing embeddings (resume-safe: chi embed frame nao chua co)
if EMBEDDINGS_PKL.exists():
    with open(EMBEDDINGS_PKL, 'rb') as f:
        store = pickle.load(f)
    print(f"Da co {len(store['ids'])} embedding tu truoc.")
else:
    store = {'ids': [], 'pe_core': []}

done_ids = set(store['ids'])

frame_re = re.compile(r'^(.+?)_frame_(\d+)\.jpg$')
grouped = defaultdict(list)
for fp in KEYFRAME_DIR.glob('*.jpg'):
    m = frame_re.match(fp.name)
    if not m:
        continue
    vid, idx = m.group(1), int(m.group(2))
    frame_id = f"{vid}:{idx}"
    if frame_id in done_ids:
        continue
    grouped[vid].append((idx, fp, frame_id))

n_pending = sum(len(v) for v in grouped.values())
print(f"Con {n_pending} keyframe can embed ({len(grouped)} video).\n")

@torch.inference_mode()
def embed_images(images):
    tensors = torch.stack([pecore_preprocess(im) for im in images]).to(device)
    if device == 'cuda':
        tensors = tensors.half()
    feats = pecore_model.encode_image(tensors)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.float().cpu().numpy()

BATCH = 32
SAVE_EVERY = 500  # frames -- ghi dinh ky de khong mat cong neu bi ngat giua chung
since_save = 0
t0 = time.time()
total_done = 0

for vname, items in sorted(grouped.items()):
    items.sort(key=lambda x: x[0])
    for bi in range(0, len(items), BATCH):
        chunk = items[bi:bi + BATCH]
        images = [Image.open(fp).convert('RGB') for _, fp, _ in chunk]
        try:
            embs = embed_images(images)
        except Exception as e:
            print(f"  LOI embed {vname} batch {bi}: {e}")
            continue
        for (idx, fp, frame_id), emb in zip(chunk, embs):
            store['ids'].append(frame_id)
            store['pe_core'].append(emb)
        total_done += len(chunk)
        since_save += len(chunk)
        if since_save >= SAVE_EVERY:
            with open(EMBEDDINGS_PKL, 'wb') as f:
                pickle.dump(store, f)
            since_save = 0
            elapsed = time.time() - t0
            rate = total_done / elapsed if elapsed > 0 else 0
            print(f"  ... {total_done}/{n_pending} frame ({rate:.1f} frame/s) -- da luu")

with open(EMBEDDINGS_PKL, 'wb') as f:
    pickle.dump(store, f)

print(f"\n=== XONG embedding: {len(store['ids'])} frame trong {EMBEDDINGS_PKL} ===")